# Gold-Set v2 — passagen-genaue Labels (n ≥ 30–50)

**Warum?** Das bisherige Gold-Set (`eval/gold_dracula.jsonl`, n=21) labelt auf **Kapitel-Ebene**.
Das hat zwei gemessene Probleme:

1. Ein Kapitel deckt viele Chunks ab → ein „Hit" ist billig, die Metrik überschätzt das Retrieval
   (und unterbewertet den passagen-genauen Reranker — Befund aus Exp 3).
2. n=21 ist zu klein für signifikante Vergleiche (Befund aus Exp 2: 4:4-Bilanz, keine Aussage möglich).

**Design-Entscheidung — Span-Labels statt Chunk-IDs:** Ein Label ist ein Textbereich
`{book_id, char_start, char_end}` im bereinigten Buchtext. Ein Retrieval-Treffer liegt vor, wenn ein
gefundener Chunk einen Gold-Span **überlappt**. Vorteil: Chunk-Indizes verschieben sich beim Re-Chunking
(andere `target_chars`), Zeichen-Offsets im bereinigten Text nicht. Damit bleibt das Gold-Set gültig,
wenn wir später am Chunking schrauben — genau das wollen wir ja messen können.

**Voraussetzungen:** `make up` (Qdrant :6333, Backend :8001) und Host-Ollama (für die Such-Helfer).

**Start:**
```bash
cd ~/LLM/book-RAG
uv run --with jupyterlab --with pandas --with matplotlib jupyter lab notebooks/gold_set_v2.ipynb
```


In [ ]:
import json
import random
import urllib.request

import pandas as pd

QDRANT_URL = "http://localhost:6333"
OLLAMA_URL = "http://localhost:11434"
COLLECTION = "book_chunks"
EMBED_MODEL = "bge-m3"

GROUP_ID = "Horror"          # Zielgruppe fuer diesen Durchgang (GoT = zweiter Durchgang)
K = 10                        # Kandidaten pro Such-Vorschau
SEED = 42                     # Reproduzierbare Stichproben

GOLD_V1_PATH = "../eval/gold_dracula.jsonl"
GOLD_V2_PATH = "../eval/gold_v2.jsonl"

pd.set_option("display.max_colwidth", 100)


def _post(url: str, body: dict) -> dict:
    req = urllib.request.Request(
        url, data=json.dumps(body).encode(), headers={"Content-Type": "application/json"}
    )
    return json.load(urllib.request.urlopen(req))


## 1. Bestandsaufnahme: alle Chunks der Gruppe laden

Wir lesen die Chunks direkt aus Qdrant (`scroll`, ohne Vektoren) — dieselben Daten, auf denen
die App sucht. Das DataFrame ist unsere Arbeitsgrundlage für Stichproben und Labeling.


In [ ]:
def load_chunks(group_id: str) -> pd.DataFrame:
    rows, offset = [], None
    while True:
        body = {
            "limit": 256, "offset": offset, "with_payload": True, "with_vector": False,
            "filter": {"must": [{"key": "group_id", "match": {"value": group_id}}]},
        }
        res = _post(f"{QDRANT_URL}/collections/{COLLECTION}/points/scroll", body)["result"]
        for p in res["points"]:
            pl = p["payload"]
            rows.append({
                "book_id": pl["book_id"], "book_title": pl.get("book_title"),
                "chapter": pl.get("chapter"), "chunk_index": pl["chunk_index"],
                "char_start": pl["char_start"], "char_end": pl["char_end"],
                "n_chars": pl["char_end"] - pl["char_start"], "text": pl["text"],
            })
        offset = res.get("next_page_offset")
        if offset is None:
            break
    return pd.DataFrame(rows).sort_values(["book_id", "chunk_index"]).reset_index(drop=True)


chunks = load_chunks(GROUP_ID)
print(f"{len(chunks)} Chunks in Gruppe '{GROUP_ID}'")
chunks[["book_title", "chapter", "chunk_index", "char_start", "char_end", "n_chars"]].head()


In [ ]:
# Verteilung: Chunks pro Kapitel — zeigt, wie grob ein Kapitel-Label ist.
per_chapter = chunks.groupby("chapter", dropna=False).size().rename("n_chunks")
print(per_chapter.describe().round(1))
per_chapter.sort_values(ascending=False).head(10)


## 2. Wie billig ist ein Kapitel-Hit? (Granularitäts-Problem quantifizieren)

Für jedes v1-Item zählen wir, wie viele Chunks seine `relevant_chapters` abdecken.
Deckt ein Label z. B. 40 Chunks ab, reicht irgendein Treffer aus 40, um als „Hit" zu zählen —
obwohl vielleicht nur 2 davon die Antwort wirklich enthalten.


In [ ]:
import re

_ROMAN_RE = re.compile(r"CHAPTER\s+([IVXLCDM]+)", re.IGNORECASE)


def chapter_roman(chapter):
    if not isinstance(chapter, str):  # None/NaN (Chunks vor der ersten Ueberschrift)
        return None
    m = _ROMAN_RE.search(chapter)
    return m.group(1).upper() if m else None


gold_v1 = [json.loads(l) for l in open(GOLD_V1_PATH, encoding="utf-8") if l.strip()]
chunks["roman"] = chunks["chapter"].map(chapter_roman)

rows = []
for item in gold_v1:
    rel = {r.upper() for r in item["relevant_chapters"]}
    covered = int((chunks["roman"].isin(rel)).sum())
    rows.append({"id": item["id"], "kapitel": ",".join(sorted(rel)), "abgedeckte_chunks": covered})
coverage = pd.DataFrame(rows)
print(f"Ø abgedeckte Chunks pro Kapitel-Label: {coverage['abgedeckte_chunks'].mean():.1f}")
print(f"(bei {len(chunks)} Chunks gesamt — so viel 'Trefferfläche' schenkt ein Kapitel-Label)")
coverage


In [ ]:
import matplotlib.pyplot as plt

COL_KAPITEL = "#2a78d6"   # Serie 1 (blau)
COL_SPAN = "#1baf7a"      # Serie 2 (aqua) — braucht direkte Wertelabels (Kontrast-Relief)


def _style_ax(ax):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_facecolor("white")
    return ax


# Trefferflaeche pro v1-Item: wie viele Chunks zaehlen unter dem Kapitel-Label als Treffer?
cov = coverage.sort_values("abgedeckte_chunks")
fig, ax = plt.subplots(figsize=(8, 5.5), dpi=110, facecolor="white")
_style_ax(ax)
ax.barh(cov["id"], cov["abgedeckte_chunks"], height=0.62, color=COL_KAPITEL)
mean_val = cov["abgedeckte_chunks"].mean()
ax.axvline(mean_val, color="#52514e", lw=1, ls="--")
ax.text(mean_val + 0.6, len(cov) - 0.6, f"Ø {mean_val:.1f}", color="#52514e", fontsize=9)
ax.axvline(2, color="#52514e", lw=1, ls=":")
ax.set_xlabel("als „relevant“ zählende Chunks (von 581)\n"
              "gepunktet: typisches Span-Label (1–2 Chunks) · gestrichelt: Ø der Kapitel-Labels")
ax.set_title("Trefferfläche der Kapitel-Labels im v1-Gold-Set", fontsize=11, loc="left")
ax.grid(axis="x", alpha=0.25)
ax.tick_params(length=0)
plt.tight_layout()
plt.show()


## 3. Sampling-Strategie für neue Items

Ziel für diesen Durchgang (Gruppe Horror/Dracula): **von 21 auf ~35 Items**, danach derselbe
Workflow für GoT. Damit das Set nicht nur „leichte" Fragen enthält, mischen wir Fragetypen:

| Typ | Anteil | Beispiel |
|---|---|---|
| `fact` — direkte Faktenfrage | ~50 % | „Was geschieht mit der Demeter in Whitby?" |
| `paraphrase` — Frage nutzt andere Wörter als der Text | ~25 % | „Wie überwindet der Graf senkrechte Mauern?" |
| `multi` — Antwort verteilt über mehrere Stellen | ~15 % | „Welche Regeln gelten für Vampire?" |
| `unanswerable` — steht nicht im Buch (`spans: []`) | ~10 % | „Wie heißt Draculas Bruder?" |

`unanswerable`-Items messen, ob die Pipeline ehrlich bleibt („Dazu steht in den Quellen nichts") —
und ob das Retrieval bei solchen Fragen wenigstens nichts Hochkonfidentes zurückgibt.

Die Stichprobe unten zieht Chunks **stratifiziert über den Buchverlauf** (Anfang/Mitte/Ende),
damit die Fragen nicht alle aus den ersten Kapiteln kommen — lies einen Chunk, schreib eine Frage
dazu (in eigenen Worten!), und notiere dir seinen Bereich.


In [ ]:
random.seed(SEED)
n_per_bucket = 4  # pro Drittel des Buchs

sample_rows = []
for name, part in chunks.groupby(pd.qcut(chunks["chunk_index"], 3, labels=["Anfang", "Mitte", "Ende"]), observed=True):
    picks = part.sample(n=min(n_per_bucket, len(part)), random_state=SEED)
    for _, r in picks.iterrows():
        sample_rows.append({"bucket": name, **r[["chapter", "chunk_index", "char_start", "char_end"]].to_dict(),
                            "preview": r["text"][:220].replace("\n", " ") + " …"})
pd.DataFrame(sample_rows)


In [ ]:
# Einen Kandidaten im Volltext lesen (chunk_index anpassen):
CHUNK_INDEX = int(chunks["chunk_index"].iloc[0])

row = chunks[chunks["chunk_index"] == CHUNK_INDEX].iloc[0]
print(f"{row['book_title']} — {row['chapter']}  (chunk {row['chunk_index']}, "
      f"Zeichen {row['char_start']}–{row['char_end']})\n")
print(row["text"])


### Optional: LLM-Entwürfe für Fragen (opt-in)

`qwen2.5` kann pro Chunk einen Frage-Entwurf liefern. **Zwei Regeln:** (1) Entwürfe sind nur
Rohmaterial — jede Frage wird manuell umformuliert und geprüft, sonst evaluieren wir später
das Modell gegen seine eigenen Formulierungen (Leakage: die Frage übernimmt Wortlaut aus dem
Chunk, und das dense Retrieval findet sie trivial). (2) Antwort immer selbst gegen den Chunk
verifizieren.


In [ ]:
GENERATE_DRAFTS = False  # bewusst opt-in; Ollama muss laufen

def draft_question(chunk_text: str, model: str = "qwen2.5:7b-instruct-q4_K_M") -> str:
    body = {
        "model": model, "stream": False,
        "messages": [{
            "role": "user",
            "content": (
                "Formuliere EINE deutsche Quizfrage zu folgendem Romanausschnitt. "
                "Die Frage darf keine Formulierungen aus dem Text übernehmen und muss "
                "ohne den Text verständlich sein. Antworte NUR mit der Frage.\n\n" + chunk_text
            ),
        }],
    }
    return _post(f"{OLLAMA_URL}/api/chat", body)["message"]["content"].strip()


if GENERATE_DRAFTS:
    for s in sample_rows[:3]:
        row = chunks[chunks["chunk_index"] == s["chunk_index"]].iloc[0]
        print(f"chunk {s['chunk_index']} ({s['chapter']}):")
        print("  Entwurf:", draft_question(row["text"]), "\n")
else:
    print("GENERATE_DRAFTS=False — Zelle übersprungen.")


## 4. Labeling-Helfer: Frage → Kandidaten → Spans wählen

Der Kern des Workflows. Für eine (selbst geschriebene) Frage zeigt `preview_search` die Top-K
der **echten** Pipeline-Suche (bge-m3 + Cosine + Gruppenfilter). Daraus wählst du die Chunks,
die die Antwort wirklich enthalten, und übernimmst deren Zeichenbereiche als Gold-Spans.

**Wichtig wegen Chunk-Overlap:** Dieselbe Passage kann am Ende von Chunk *i* **und** am Anfang
von Chunk *i+1* stehen. Prüfe deshalb immer auch die Nachbar-Chunks (`show_neighbors`) und labe
die Passage, nicht den Chunk — im Zweifel beide überlappenden Bereiche als einen Span zusammenfassen.
Benachbarte/überlappende Spans werden beim Hinzufügen automatisch verschmolzen.


In [ ]:
def _l2(v):
    s = sum(x * x for x in v) ** 0.5
    return [x / s for x in v] if s else v


def preview_search(question: str, k: int = K) -> pd.DataFrame:
    emb = _post(f"{OLLAMA_URL}/api/embed", {"model": EMBED_MODEL, "input": [question]})["embeddings"][0]
    body = {
        "query": _l2(emb), "limit": k, "with_payload": True,
        "filter": {"must": [{"key": "group_id", "match": {"value": GROUP_ID}}]},
    }
    pts = _post(f"{QDRANT_URL}/collections/{COLLECTION}/points/query", body)["result"]["points"]
    return pd.DataFrame([{
        "rank": i + 1, "score": round(p["score"], 3),
        "chapter": p["payload"].get("chapter"), "chunk_index": p["payload"]["chunk_index"],
        "char_start": p["payload"]["char_start"], "char_end": p["payload"]["char_end"],
        "preview": p["payload"]["text"][:160].replace("\n", " ") + " …",
    } for i, p in enumerate(pts)])


def show_neighbors(chunk_index: int, radius: int = 1) -> pd.DataFrame:
    sel = chunks[chunks["chunk_index"].between(chunk_index - radius, chunk_index + radius)]
    out = sel[["chapter", "chunk_index", "char_start", "char_end"]].copy()
    out["preview"] = sel["text"].str[:160].str.replace("\n", " ", regex=False) + " …"
    return out


preview_search("Warum reist Jonathan Harker nach Transsilvanien?")


In [ ]:
show_neighbors(2)

## 5. Gold-Items erfassen

`add_item` sammelt neue Items in `gold_v2` (in-memory) und validiert:
eindeutige ID, Spans innerhalb des Buchs, Antwort nicht leer. Benachbarte oder überlappende
Spans werden automatisch zu einem verschmolzen. `spans=[]` ist nur für `qtype="unanswerable"` erlaubt.


In [ ]:
gold_v2: list[dict] = []

_book_extent = chunks.groupby("book_id")["char_end"].max().to_dict()
_default_book = chunks["book_id"].iloc[0]


def _merge_spans(spans: list[tuple[int, int]]) -> list[dict]:
    merged: list[list[int]] = []
    for start, end in sorted(spans):
        if merged and start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return [{"char_start": s, "char_end": e} for s, e in merged]


def add_item(item_id: str, question: str, answer: str, spans: list[tuple[int, int]],
             qtype: str = "fact", chapters: list[str] | None = None,
             book_id: str | None = None, question_en: str = "", note: str = "",
             source: str = "manual") -> None:
    assert qtype in {"fact", "paraphrase", "multi", "unanswerable"}, f"unbekannter qtype: {qtype}"
    assert item_id not in {g["id"] for g in gold_v2}, f"ID {item_id} existiert schon"
    assert question.strip() and answer.strip(), "Frage/Antwort darf nicht leer sein"
    book = book_id or _default_book
    if qtype == "unanswerable":
        assert not spans, "unanswerable-Items haben keine Spans"
    else:
        assert spans, "mindestens ein Span noetig (oder qtype='unanswerable')"
        extent = _book_extent[book]
        for s, e in spans:
            assert 0 <= s < e <= extent, f"Span ({s},{e}) ausserhalb des Buchs (0..{extent})"
    gold_v2.append({
        "id": item_id, "question": question.strip(), "question_en": question_en.strip(),
        "answer": answer.strip(), "book_id": book, "spans": _merge_spans(spans),
        "qtype": qtype, "chapters": chapters or [], "note": note, "source": source,
    })
    print(f"ok — {len(gold_v2)} Items erfasst")


# Beispiel-Workflow (an d01 angelehnt): Frage suchen, Kandidat MANUELL pruefen
# (Volltext lesen!), dann dessen Bereich als Span uebernehmen.
_demo_q = "Aus welchem Grund faehrt der junge Anwalt zu Beginn nach Osteuropa?"
_demo_top = preview_search(_demo_q).iloc[0]  # hier: Rank 1 nach Sichtung als korrekt bestaetigt

add_item(
    "v2_demo",
    question=_demo_q,
    answer="Er soll fuer Graf Dracula den Kauf eines Anwesens in England (Carfax) abwickeln.",
    spans=[(int(_demo_top["char_start"]), int(_demo_top["char_end"]))],
    qtype="paraphrase", chapters=["I", "II"], source="manual",
)
gold_v2[-1]


## 6. v1-Items übernehmen (Spans nachtragen)

Die 21 bestehenden Items bleiben wertvoll — sie brauchen nur Spans. Workflow pro Item:
Zelle unten zeigt die Suche für das v1-Item plus alle Chunks seiner Gold-Kapitel;
daraus die tatsächlich antwort-tragenden Bereiche als Spans in `V1_SPANS` eintragen.
(Chapters übernehmen wir mit — so bleibt die alte Metrik zum Vergleich berechenbar.)


In [ ]:
V1_INDEX = 0  # 0..20: welches v1-Item gerade gelabelt wird

item = gold_v1[V1_INDEX]
print(json.dumps(item, ensure_ascii=False, indent=2))
print("\n— Top-Kandidaten der Suche: —")
display(preview_search(item["question"]))
rel = {r.upper() for r in item["relevant_chapters"]}
print("— alle Chunks der Gold-Kapitel: —")
sel = chunks[chunks["roman"].isin(rel)]
out = sel[["chapter", "chunk_index", "char_start", "char_end"]].copy()
out["preview"] = sel["text"].str[:120].str.replace("\n", " ", regex=False) + " …"
out


In [ ]:
# Nach dem Sichten: Spans je v1-ID hier sammeln und dann ueberfuehren, z. B.
# V1_SPANS = {"d01": [(0, 1903), (5211, 7100)], "d02": [(31000, 32900)]}
V1_SPANS: dict[str, list[tuple[int, int]]] = {}

for item in gold_v1:
    if item["id"] in V1_SPANS and item["id"] not in {g["id"] for g in gold_v2}:
        add_item(item["id"], item["question"], item["answer"], V1_SPANS[item["id"]],
                 qtype="fact", chapters=item["relevant_chapters"],
                 question_en=item.get("question_en", ""), note=item.get("note", ""), source="v1")
print(f"{len(V1_SPANS)} v1-Items mit Spans versehen.")


## 7. Smoke-Eval: so rechnet die neue Metrik

Span-basierte Hit-Rate@k und MRR über die bisher erfassten Items: ein Treffer ist ein
gefundener Chunk desselben Buchs, dessen Bereich einen Gold-Span überlappt.
`unanswerable`-Items sind hier ausgenommen (für sie zählt später die Antwort-Ehrlichkeit,
nicht das Retrieval).


In [ ]:
def span_hit(point_row: dict, item: dict) -> bool:
    if point_row.get("book_id") not in (None, item["book_id"]):
        return False
    return any(point_row["char_start"] < s["char_end"] and s["char_start"] < point_row["char_end"]
               for s in item["spans"])


def smoke_eval(items: list[dict], k: int = 8) -> pd.DataFrame:
    rows = []
    for item in items:
        if item["qtype"] == "unanswerable":
            continue
        cands = preview_search(item["question"], k=k)
        hits = [r + 1 for r, c in cands.iterrows() if span_hit(c.to_dict(), item)]
        rows.append({"id": item["id"], "qtype": item["qtype"],
                     "erster_hit_rank": hits[0] if hits else None,
                     "hit@1": bool(hits and hits[0] == 1), f"hit@{k}": bool(hits),
                     "rr": 1 / hits[0] if hits else 0.0})
    df = pd.DataFrame(rows)
    if len(df):
        print(f"Hit-Rate@1: {df['hit@1'].mean():.2f}   Hit-Rate@{k}: {df[f'hit@{k}'].mean():.2f}   "
              f"MRR: {df['rr'].mean():.3f}   (n={len(df)})")
    return df


smoke_eval(gold_v2)


### Grafik: Vorher/Nachher — Kapitel- vs. Span-Metrik auf denselben Fragen

Der eigentliche Punkt des v2-Sets, als paarweiser Vergleich: für jedes Item, das **beide**
Label-Arten trägt (Kapitel aus v1, Spans aus v2), werten wir **dieselben** Suchergebnisse
zweimal aus. Erwartung: die Kapitel-Metrik fällt systematisch zu gnädig aus — die Differenz
der Balken ist genau die Überschätzung, die wir loswerden wollen. (Aussagekräftig ab ~20
gelabelten Items; mit wenigen Items ist es nur eine Illustration.)


In [ ]:
def paired_metrics(items: list[dict], k: int = 8) -> pd.DataFrame | None:
    eligible = [i for i in items if i["qtype"] != "unanswerable" and i["chapters"] and i["spans"]]
    if not eligible:
        print("Noch keine Items mit Kapitel- UND Span-Labels erfasst.")
        return None
    rows = []
    for item in eligible:
        cands = preview_search(item["question"], k=k)
        rel = {r.upper() for r in item["chapters"]}
        ch_hits = [r + 1 for r, c in cands.iterrows() if chapter_roman(c["chapter"]) in rel]
        sp_hits = [r + 1 for r, c in cands.iterrows() if span_hit(c.to_dict(), item)]
        rows.append({
            "kap_h1": bool(ch_hits and ch_hits[0] == 1), "kap_hk": bool(ch_hits),
            "kap_rr": 1 / ch_hits[0] if ch_hits else 0.0,
            "spa_h1": bool(sp_hits and sp_hits[0] == 1), "spa_hk": bool(sp_hits),
            "spa_rr": 1 / sp_hits[0] if sp_hits else 0.0,
        })
    df = pd.DataFrame(rows)
    return pd.DataFrame({
        "Kapitel-Label (v1)": [df["kap_h1"].mean(), df["kap_hk"].mean(), df["kap_rr"].mean()],
        "Span-Label (v2)": [df["spa_h1"].mean(), df["spa_hk"].mean(), df["spa_rr"].mean()],
    }, index=["Hit-Rate@1", "Hit-Rate@8", "MRR"])


paired = paired_metrics(gold_v2)
if paired is not None:
    n_items = len([i for i in gold_v2 if i["qtype"] != "unanswerable" and i["chapters"] and i["spans"]])
    x = range(len(paired.index))
    w = 0.36
    fig, ax = plt.subplots(figsize=(7, 4), dpi=110, facecolor="white")
    _style_ax(ax)
    b1 = ax.bar([i - w / 2 for i in x], paired["Kapitel-Label (v1)"], width=w,
                color=COL_KAPITEL, label="Kapitel-Label (v1)")
    b2 = ax.bar([i + w / 2 for i in x], paired["Span-Label (v2)"], width=w,
                color=COL_SPAN, label="Span-Label (v2)")
    for bars in (b1, b2):
        ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=9, color="#0b0b0b")
    ax.set_xticks(list(x), paired.index)
    ax.set_ylim(0, 1.12)
    ax.set_title(f"Dieselben Suchergebnisse, zwei Maßstäbe (n={n_items})", fontsize=11, loc="left")
    ax.grid(axis="y", alpha=0.25)
    ax.tick_params(length=0)
    ax.legend(frameon=False, fontsize=9)
    plt.tight_layout()
    plt.show()


## 8. Export

Schreibt `eval/gold_v2.jsonl` (ein Item pro Zeile). `SAVE=True` setzen, wenn der Stand
gesichert werden soll — die Zelle überschreibt eine bestehende Datei **nicht**, sondern
verweigert dann mit Hinweis (bewusst, damit kein kuratierter Stand verloren geht;
alte Datei vorher umbenennen oder Pfad ändern).

## Nächster Schritt (außerhalb dieses Notebooks)

`eval/retrieval_eval.py` bekommt einen `--gold-v2`-Modus: gleiche Retrieve-Schleife, Hit =
Span-Overlap statt Kapitel-Match (Funktion `span_hit` oben lässt sich 1:1 übernehmen). Die
Kapitel-Metrik bleibt als Vergleich erhalten — dann sehen wir schwarz auf weiß, wie stark
Kapitel-Labels das Retrieval überschätzt haben, und `answer_eval.py`/Reranker-Experimente
laufen ab dann gegen das faire Set.


In [ ]:
import os

SAVE = False

if SAVE:
    assert gold_v2, "nichts zu exportieren"
    assert not os.path.exists(GOLD_V2_PATH), f"{GOLD_V2_PATH} existiert schon — erst umbenennen."
    with open(GOLD_V2_PATH, "w", encoding="utf-8") as f:
        for item in gold_v2:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"{len(gold_v2)} Items → {GOLD_V2_PATH}")
else:
    print(f"SAVE=False — {len(gold_v2)} Items im Speicher, nichts geschrieben.")
